In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import math
import random

In [2]:
class Value:
    """ stores a single scalar value and its gradient """

    def __init__(self, data, _children=(), _op='', label='' ):
        self.data = data
        self.grad = 0
        # internal variables used for autograd graph construction
        self._backward = lambda: None
        self._prev = set(_children)
        self.label = label
        self._op = _op # the op that produced this node, for graphviz / debugging / etc

    def __add__(self, other):
        #if other is not a Value, convert it to a Value(e.g. if it's a number like 2.0)
        other  = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        
        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        out._backward = _backward
        return out
    def __mul__(self, other):
        other  = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out
    
    def __pow__(self, other):
        assert isinstance(other, (int, float)), "only supporting int/float powers for now"
        out = Value(self.data ** other, (self,), f'**{other}')
        
        def _backward():
            self.grad += (other * self.data**(other-1)) * out.grad
        out._backward = _backward
        return out
    
    #refer the tanh function from the 2.png file
    def tanh(self):
        x = self.data
        t = (math.exp(2*x) - 1) / (math.exp(2*x) + 1)
        out = Value(t, (self, ), 'tanh')
        
        def _backward():
            self.grad += (1 - t**2) * out.grad
        out._backward = _backward
        return out
    def exp(self):
        x = self.data
        out = Value(math.exp(x), (self,), 'exp')
        
        def _backward():
            self.grad += out.grad * out.data
        out._backward = _backward
        return out
    def backward(self):
        # traverse all the previous nodes in reverse topological order
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        
        self.grad = 1.0
        # go one node at a time and call backward on it
        for node in reversed(topo):
            node._backward()
            
    def __rmul__(self, other): #other is a number like 2.0(other * self ---> reverse multiplication)
        return self * other #this will call the __mul__ function
    
    def __truediv__(self, other): #self / other
        return self * other**-1
    
    def __neg__(self): #-self
        return self * -1
    
    def __sub__(self, other): #self - other
        return self + (-other)
    
    def __repr__(self):
        return f"Value(data = {self.data})"        
        

In [3]:
from graphviz import Digraph

def trace(root):
    # builds a set of all nodes and edges in a graph
    nodes, edges = set(), set()
    def build(v):
        if v not in nodes:
            nodes.add(v)
            for child in v._prev:
                edges.add((child, v))
                build(child)
    build(root)
    return nodes, edges

def draw_dot(root):
    dot = Digraph(format='svg', graph_attr={'rankdir': 'LR'}) # LR = left to right
    
    nodes, edges = trace(root)
    for n in nodes:
        uid = str(id(n))
        # for any value in the graph, create a rectangular ('record') node for it
        dot.node(name = uid, label = "{ %s | data %.4f | grad %.4f }" % (n.label, n.data, n.grad), shape='record')
        if n._op:
            # if this value is a result of some operation, create an op node for it
            dot.node(name = uid + n._op, label = n._op)
            # and connect this node to it
            dot.edge(uid + n._op, uid)
            
    for n1, n2 in edges:
        # connect n1 to the op node of n2
        dot.edge(str(id(n1)), str(id(n2)) + n2._op)
        
    return dot

In [4]:
class Neuron:
    
    def __init__(self, nin): # nin is the number of inputs to the neuron
        #this will create a list of weights for each input, and a bias term, all initialized to random values between -1 and 1
        self.w = [Value(random.uniform(-1,1)) for _ in range(nin)]
        self.b = Value(random.uniform(-1,1))
        
    def __call__(self, x):
        # w * x + b
        out = sum((wi*xi for wi, xi in zip(self.w, x)),self.b)
        return out.tanh()
    
    def parameters(self):
        return self.w + [self.b]# this will return a list of all the parameters in the neuron, which are the weights and the bias
    
#class Layer is a collection of neurons, and the output of the layer is the output of each neuron in the layer
class Layer:
    #nout is the number of neurons in the layer, and nin is the number of inputs to each neuron
    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for i in range(nout)]
        
    #this call function will take a list of inputs, and return a list of outputs, one for each neuron in the layer
    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs # if there's only one neuron, return the output of that neuron, otherwise return a list of outputs for each neuron in the layer
    
    def parameters(self):
        return [p for neuron in self.neurons for p in neuron.parameters()] # this will return a list of all the parameters in the layer, 
                                                                           # which are the parameters of each neuron in the layer
class MLP:
    #nouts is a list of the number of neurons in each layer, including the output layer, 
    #but not including the input layer, which is given by nin
    def __init__(self, nin, nouts):
        #sz is a list of the number of neurons in each layer, including the input layer, which is given by nin, and the output layer, which is given by nouts
        sz = [nin] + nouts
        #in this we will create a list of layers, where each layer is a Layer object, 
        #and the number of neurons in each layer is given by the corresponding value in sz
        self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]
        
    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x
    
    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()] # this will return a list of all the parameters in the network,
                                                                        # which are the parameters of each layer in the network
        
        
            

In [5]:
x = [2.0, 3.0, -1.0]
n = MLP(3, [4, 4, 1])
n(x)

Value(data = -0.7099020956127432)

In [6]:
# taking an example 
xs = [
    [2.0, 3.0, -1.0],
    [3.0, -1.0, 0.5],
    [0.5, 1.0, 1.0],
    [1.0, 1.0, -1.0],
]
ys = [1.0, -1.0, -1.0, 1.0]



In [ ]:
#applying the training loop to the network
#Gradient descent is an optimization algorithm used to minimize the loss function by iteratively updating the parameters 
#of the model in the direction of the negative gradient of the loss function with respect to the parameters.

for k in range(30):
    #forward pass
    ypreds = [n(x) for x in xs]
    loss = sum(((ypred - y)**2 for ypred, y in zip(ypreds, ys)), Value(0))
    
    #backward pass
    for p in n.parameters():
        p.grad = 0.0 # set the gradient of all parameters to zero before backward pass
    loss.backward()
    
    #update
    for p in n.parameters():
        p.data += -0.05 * p.grad
    
    print(k, loss.data)

0 5.987245625331864
1 4.536789970063991
2 3.947188011135438
3 3.701410408905544
4 3.3767373213980933
5 2.919847274699202
6 2.3537572325014375
7 1.7599059368209509
8 1.2415734305056896
9 0.851884308787872
10 0.5918079940359365
11 0.42664239454098724
12 0.32126514413377727
13 0.25170734383279214
14 0.2038432649593636
15 0.16957169355189503
16 0.14415718311066103
17 0.12473679711956866
18 0.10951410184205196
19 0.09732083229072713
20 0.08737199808366622
21 0.07912448550454909
22 0.07219263867487018
23 0.06629622766730177
24 0.06122741918707136
25 0.05682923780218492
26 0.05298117254453003
27 0.049589342684383
28 0.04657964188509211
29 0.043892870418195284


In [8]:
ypreds

[Value(data = 0.9084356721137056),
 Value(data = -0.9179371565631934),
 Value(data = -0.8681277155804048),
 Value(data = 0.8933030712440606)]